In [ ]:
# ==========================================
# Weather Prediction: Rain Classification
# Models: Logistic Regression, Decision Tree, Random Forest
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os

# 1. Load Data
file_name = "weather_forecast_data.csv"

if os.path.exists(file_name):
    data = pd.read_csv(file_name)
    print(f"✅ Dataset loaded successfully from {file_name}\n")
else:
    print(f"⚠️ '{file_name}' not found. Using sample data for demo...\n")
    np.random.seed(42)
    data = pd.DataFrame({
        'Temperature': np.random.uniform(10, 35, 2500),
        'Humidity': np.random.uniform(30, 100, 2500),
        'Wind_Speed': np.random.uniform(0, 20, 2500),
        'Cloud_Cover': np.random.uniform(0, 100, 2500),
        'Pressure': np.random.uniform(980, 1050, 2500),
        'Rain': np.random.choice(['rain', 'no rain'], 2500)
    })

# 2. Data Preprocessing
data = data.dropna()
X = data.drop(columns=['Rain'])
y = data['Rain']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train Models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print("="*50)
    print(f"--- {name} Results ---")
    print("="*50)

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("\n")

# 4. Compare Results
print("="*50)
print("--- Model Comparison Summary ---")
print("="*50)
for name, acc in results.items():
    print(f"{name}: {acc:.4f} Accuracy")

# 5. Visualization (Confusion Matrix for Random Forest)
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix - {best_model_name} (Best Model)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()